Séance : 2  
Nom : SANNI Farhanou Kankou  
Porteur : P1  
Etapes : 2.1 (le découpage), 2.4 (l'étanchéité), 2.7 (le tableau d'essai)

## Chargement et paramètres

In [1]:
import os
import time
import numpy as np
import pandas as pd
from datetime import date
from sklearn.model_selection import KFold, StratifiedKFold

train = pd.read_csv("train.csv")        # sinistres avec la cible loss
test = pd.read_csv("a_estimer.csv")     # sinistres sans loss

y = train["loss"].to_numpy()                       # la cible
cat = [f"cat{i}" for i in range(1, 117)]           # 116 variables catégorielles
cont = [f"cont{i}" for i in range(1, 15)]          # 14 variables continues
variables = cat + cont                             # les 130 variables d'entrée

# PARAMÈTRES du protocole : on les change ICI, et seulement ici.
# C'est ce qui rend le code "réparable" (une cellule à modifier, pas tout le pipeline).
N_PLIS = 5          # nombre de parts du découpage
GRAINE = 0          # fixe le hasard : mêmes plis à chaque exécution
N_TRANCHES = 10     # tranches de loss utilisées pour stratifier (10 = déciles)

# MAE : moyenne des écarts en valeur absolue (la métrique retenue)
def mae(y_vrai, c):
    return np.mean(np.abs(y_vrai - c))

## 2.1 Le découpage

In [2]:
# La fonction de découpage : on la réutilise telle quelle dans tout le projet.
def decouper(y, n_plis=N_PLIS, graine=GRAINE, n_tranches=N_TRANCHES):
    # pd.qcut range chaque sinistre dans une tranche de loss (0 = les plus petits).
    tranches = pd.qcut(y, q=n_tranches, labels=False)
    # StratifiedKFold : chaque pli reçoit la même proportion de chaque tranche.
    sp = StratifiedKFold(n_splits=n_plis, shuffle=True, random_state=graine)
    # list(...) transforme le résultat en liste de paires (tr, va) :
    # tr = positions des lignes d'entraînement, va = positions des lignes de validation.
    return list(sp.split(y, tranches))

plis = decouper(y)
for k, (tr, va) in enumerate(plis, start=1):
    print(f"pli {k} : {len(tr)} lignes en entraînement, {len(va)} en validation")

pli 1 : 120523 lignes en entraînement, 30131 en validation
pli 2 : 120523 lignes en entraînement, 30131 en validation
pli 3 : 120523 lignes en entraînement, 30131 en validation
pli 4 : 120523 lignes en entraînement, 30131 en validation
pli 5 : 120524 lignes en entraînement, 30130 en validation


### Ce que chaque pli contient : découpage simple contre découpage stratifié

In [3]:
# Pour chaque pli de validation : moyenne, médiane, percentile 99 et maximum de loss.
def profil(decoupes, nom):
    lignes = []
    for k, (tr, va) in enumerate(decoupes, start=1):
        lignes.append({
            "découpage": nom, "pli": k,
            "moyenne": y[va].mean(),
            "médiane": np.median(y[va]),
            "percentile 99": np.quantile(y[va], 0.99),
            "maximum": y[va].max(),
        })
    return pd.DataFrame(lignes)

# Découpage simple : au hasard, sans tenir compte de loss.
simple = list(KFold(n_splits=N_PLIS, shuffle=True, random_state=GRAINE).split(y))

print(pd.concat([profil(simple, "simple"), profil(plis, "stratifié")]).round(0).to_string(index=False))

découpage  pli  moyenne  médiane  percentile 99  maximum
   simple    1   3032.0   2102.0        13601.0  43707.0
   simple    2   3056.0   2113.0        14180.0 106863.0
   simple    3   3062.0   2138.0        14285.0 121012.0
   simple    4   3015.0   2108.0        13691.0  67667.0
   simple    5   3029.0   2123.0        14169.0  85924.0
stratifié    1   3046.0   2116.0        14293.0 121012.0
stratifié    2   3030.0   2115.0        13605.0 106863.0
stratifié    3   3046.0   2116.0        14212.0  79624.0
stratifié    4   3042.0   2116.0        14156.0  67667.0
stratifié    5   3031.0   2116.0        13870.0  51128.0


### La stabilité du score : l'écart entre plis, sur 20 tirages

In [4]:
# On note, pour chaque pli, la MAE de la constante médiane (calculée sur l'entraînement,
# notée sur la validation), puis on regarde de combien ce score varie d'un pli à l'autre.
def ecart_entre_plis(methode, graine):
    if methode == "simple":
        decoupes = KFold(n_splits=N_PLIS, shuffle=True, random_state=graine).split(y)
    else:
        decoupes = decouper(y, graine=graine)
    scores = [mae(y[va], np.median(y[tr])) for tr, va in decoupes]
    return np.std(scores)        # écart-type : de combien le score bouge entre plis

# On répète avec 20 graines différentes : une seule pourrait donner un résultat dû au hasard.
for methode in ["simple", "stratifié"]:
    ecarts = [ecart_entre_plis(methode, g) for g in range(20)]
    print(f"{methode:10s} écart-type de la MAE entre plis : {np.mean(ecarts):.2f}")

simple     écart-type de la MAE entre plis : 12.61
stratifié  écart-type de la MAE entre plis : 4.91


## 2.4 L'étanchéité

### Contrôle 1 : la structure du découpage

In [5]:
ids = train["id"].to_numpy()
fois_en_validation = np.zeros(len(y), dtype=int)   # combien de fois chaque ligne est en validation

for k, (tr, va) in enumerate(plis, start=1):
    # np.intersect1d donne les éléments communs à deux listes. Attendu : aucun.
    # assert arrête le programme avec le message si la condition est fausse.
    assert len(np.intersect1d(tr, va)) == 0, f"pli {k} : lignes communes entre entraînement et validation"
    assert len(np.intersect1d(ids[tr], ids[va])) == 0, f"pli {k} : id communs entre entraînement et validation"
    fois_en_validation[va] += 1

# Chaque ligne doit être en validation exactement une fois : ni oubliée, ni notée deux fois.
assert (fois_en_validation == 1).all(), "une ligne n'est pas en validation exactement une fois"
print("OK : plis disjoints, aucun id commun, chaque ligne validée exactement une fois")

OK : plis disjoints, aucun id commun, chaque ligne validée exactement une fois


### Contrôle 2 : des lignes identiques de part et d'autre du découpage ?

In [6]:
# Une empreinte numérique par ligne : deux lignes identiques ont la même empreinte.
h = pd.util.hash_pandas_object(train[variables], index=False).to_numpy()

for k, (tr, va) in enumerate(plis, start=1):
    # np.isin(A, B) : True pour chaque élément de A qui se trouve aussi dans B.
    deja_vues = int(np.isin(h[va], h[tr]).sum())
    print(f"pli {k} : {deja_vues} lignes de validation identiques à une ligne d'entraînement")

pli 1 : 0 lignes de validation identiques à une ligne d'entraînement
pli 2 : 0 lignes de validation identiques à une ligne d'entraînement
pli 3 : 0 lignes de validation identiques à une ligne d'entraînement
pli 4 : 0 lignes de validation identiques à une ligne d'entraînement
pli 5 : 0 lignes de validation identiques à une ligne d'entraînement


### Contrôle 3 : le test de la cible mélangée

**L'idée :** on mélange au hasard les montants `loss` entre les sinistres. Les variables n'ont alors **plus aucun lien** avec la cible. Une procédure étanche ne peut donc pas faire mieux que la constante médiane : elle n'a rien à apprendre. Si le score passe nettement **sous** ce plancher, de l'information circule quelque part.


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

def fabriquer_modele():
    # Lettre -> nombre. Une modalité jamais vue à l'apprentissage reçoit -1 au lieu de planter.
    encodeur = ColumnTransformer(
        [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat)],
        remainder="passthrough",       # les 14 continues passent sans changement
    )
    # make_pipeline enchaîne l'encodeur et le modèle : un seul .fit() apprend les deux,
    # sur les lignes d'entraînement SEULEMENT.
    return make_pipeline(
        encodeur,
        HistGradientBoostingRegressor(loss="absolute_error", max_iter=50, random_state=0),
    )

def mae_en_validation(y_cible, decoupes):
    scores = []
    for tr, va in decoupes:
        modele = fabriquer_modele()
        modele.fit(train.iloc[tr][variables], y_cible[tr])        # apprend sur l'entraînement
        pred = modele.predict(train.iloc[va][variables])          # prédit la validation
        scores.append(mae(y_cible[va], pred))
    return np.mean(scores), np.std(scores)

In [8]:
# np.random.default_rng(0).permutation(y) : mêmes montants, mais attribués au hasard.
y_melange = np.random.default_rng(0).permutation(y)

debut = time.time()
mae_modele, ecart_modele = mae_en_validation(y_melange, plis)

# Plancher sur ce même jeu mélangé : la constante médiane (calculée sur l'entraînement).
plancher_melange = np.mean([mae(y_melange[va], np.median(y_melange[tr])) for tr, va in plis])

print(f"MAE du modèle sur la cible mélangée : {mae_modele:.1f}  (écart-type {ecart_modele:.1f})")
print(f"MAE de la constante médiane          : {plancher_melange:.1f}")
print(f"écart (modèle - plancher)            : {mae_modele - plancher_melange:+.1f}")
print(f"durée : {time.time() - debut:.0f} s")

MAE du modèle sur la cible mélangée : 1810.7  (écart-type 12.2)
MAE de la constante médiane          : 1810.5
écart (modèle - plancher)            : +0.3
durée : 60 s


## 2.7 Le tableau d'essai

In [9]:
FICHIER_JOURNAL = "journal_essais.tsv"
COLONNES = ["n", "date", "auteur", "idée", "découpage", "encodage", "modèle",
            "MAE", "écart-type MAE", "RMSE", "durée (s)", "verdict"]

def charger_journal():
    # Si le fichier existe on le lit, sinon on repart d'un tableau vide.
    if os.path.exists(FICHIER_JOURNAL):
        return pd.read_csv(FICHIER_JOURNAL, sep="\t")
    return pd.DataFrame(columns=COLONNES)

def consigner(idee, decoupage, encodage, modele, mae_moyenne, ecart_type=None,
              rmse=None, duree=None, verdict="", auteur="Farhane", date_essai=None):
    """Ajoute UNE ligne au journal et l'enregistre. À appeler après chaque essai, réussi ou non."""
    journal = charger_journal()
    ligne = {
        "n": len(journal) + 1,
        "date": date_essai or date.today().isoformat(),
        "auteur": auteur, "idée": idee, "découpage": decoupage,
        "encodage": encodage, "modèle": modele,
        "MAE": mae_moyenne, "écart-type MAE": ecart_type,
        "RMSE": rmse, "durée (s)": duree, "verdict": verdict,
    }
    journal = pd.concat([journal, pd.DataFrame([ligne])], ignore_index=True) if len(journal) else pd.DataFrame([ligne])
    journal.to_csv(FICHIER_JOURNAL, sep="\t", index=False)
    return journal

### Les premiers essais : les trois constantes de référence

Ces trois lignes viennent d'une mesure réelle faite à l'étape 1, en validation croisée à 5 plis avec un `KFold` simple (graine 0), avant le choix du découpage stratifié. Elles sont notées comme telles. Le code ne les ajoute que si le journal est vide, donc relancer la cellule ne crée pas de doublons.

In [10]:
if charger_journal().empty:
    consigner("plancher : constante = moyenne de loss", "KFold simple 5 plis", "aucun",
              "constante (moyenne)", 1968.2, rmse=2914.4,
              verdict="meilleure en RMSE, pas en MAE", date_essai="étape 1")
    consigner("plancher : constante = médiane de loss", "KFold simple 5 plis", "aucun",
              "constante (médiane)", 1810.5, rmse=3057.1,
              verdict="meilleure MAE : score de référence à battre", date_essai="étape 1")
    consigner("plancher : constante = moyenne géométrique", "KFold simple 5 plis", "aucun",
              "constante (moy. géométrique)", 1811.3, rmse=3039.2,
              verdict="équivalente à la médiane en MAE (écart 0,8, dans le bruit)", date_essai="étape 1")

print(charger_journal().to_string(index=False))

 n    date  auteur                                       idée           découpage encodage                       modèle    MAE  écart-type MAE   RMSE  durée (s)                                                    verdict
 1 étape 1 Farhane     plancher : constante = moyenne de loss KFold simple 5 plis    aucun          constante (moyenne) 1968.2             NaN 2914.4        NaN                              meilleure en RMSE, pas en MAE
 2 étape 1 Farhane     plancher : constante = médiane de loss KFold simple 5 plis    aucun          constante (médiane) 1810.5             NaN 3057.1        NaN                meilleure MAE : score de référence à battre
 3 étape 1 Farhane plancher : constante = moyenne géométrique KFold simple 5 plis    aucun constante (moy. géométrique) 1811.3             NaN 3039.2        NaN équivalente à la médiane en MAE (écart 0,8, dans le bruit)


### Consigner un nouvel essai

Après chaque essai, une seule cellule, **même si l'essai échoue** :

```python
consigner(
    idee="premier modèle, encodage ordinal",
    decoupage="StratifiedKFold 5 plis (déciles)",
    encodage="OrdinalEncoder, modalités inconnues = -1",
    modele="HistGradientBoostingRegressor (loss=absolute_error)",
    mae_moyenne=...,        # à lire dans la sortie de l'essai
    ecart_type=...,
    duree=...,
    verdict="meilleur que le plancher de ... points",
)
```

Un verdict d'échec s'écrit tel quel, par exemple « n'a pas battu le plancher » ou « score trop beau : fuite trouvée et corrigée ».

In [11]:
# Lecture du journal : du meilleur (MAE la plus basse) au moins bon, avec l'écart au plancher.
PLANCHER_MAE = 1810.5       # à remplacer par la valeur de l'étape 2.3 (plancher de contrôle)

journal = charger_journal()
journal["écart au plancher"] = journal["MAE"] - PLANCHER_MAE
colonnes = ["n", "idée", "découpage", "modèle", "MAE", "écart au plancher", "verdict"]
print(journal.sort_values("MAE")[colonnes].round(1).to_string(index=False))

 n                                       idée           découpage                       modèle    MAE  écart au plancher                                                    verdict
 2     plancher : constante = médiane de loss KFold simple 5 plis          constante (médiane) 1810.5                0.0                meilleure MAE : score de référence à battre
 3 plancher : constante = moyenne géométrique KFold simple 5 plis constante (moy. géométrique) 1811.3                0.8 équivalente à la médiane en MAE (écart 0,8, dans le bruit)
 1     plancher : constante = moyenne de loss KFold simple 5 plis          constante (moyenne) 1968.2              157.7                              meilleure en RMSE, pas en MAE
